In [ ]:
import numpy as np
from datetime import datetime

node_ts_path = "node_timestamps.txt"
all_ts = []
per_node = []

with open(node_ts_path, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            per_node.append([])
            continue
        ts = []
        for x in line.split(","):
            if "." in x:
                ts.append(datetime.strptime(x, "%Y-%m-%d %H:%M:%S.%f"))
            else:
                ts.append(datetime.strptime(x, "%Y-%m-%d %H:%M:%S"))
        ts.sort()
        per_node.append(ts)
        all_ts.extend(ts)

if all_ts:
    global_min = min(all_ts)
    global_max = max(all_ts)
else:
    global_min = 0
    global_max = 0

def norm(t):
    if global_max == global_min:
        return 0.0
    return (t - global_min) / (global_max - global_min)

num_bins = 10
bin_edges_norm = [i / num_bins for i in range(num_bins + 1)]
binned_attr = np.zeros((len(per_node), num_bins), dtype=int)



for i, ts_list in enumerate(per_node):
    for ts in ts_list:
        for j in range(num_bins):
            if bin_edges_norm[j] <= norm(ts) < bin_edges_norm[j + 1]:
                binned_attr[i, j] += 1

binned_attr[binned_attr > 0] = 1

binned_attr_path = f"times_bin_{num_bins}.txt"
with open(binned_attr_path, "w") as f:
    for i in range(len(per_node)):
        line = ",".join(str(x) for x in binned_attr[i])
        f.write(line + "\n")